# 7. Export a query as PFB

`picsure::exportAsPFB(session, query, path)` runs a query server-side and writes the result as a [PFB](https://github.com/uc-cdis/pypfb) (Portable Format for Bioinformatics) file — Avro under the hood. PFB is the export format BDC uses for shipping cohorts to downstream analysis workspaces.

PFB export requires the Python `picsure[pfb]` optional dependency. If the Python env was provisioned without it, `exportAsPFB()` raises a `picsureError` pointing at the install hint.

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
session <- picsure::connect(
  platform = picsure::Platform$BDC_AUTHORIZED,
  token    = my_token
)

## Load a saved query and confirm the cohort size

Replace the UUID below with one of your own saved-query IDs.

In [ ]:
query <- picsure::loadQueryByID(session, "ea5f32d2-2ca8-4820-a96a-41239af75d38")
query_results <- picsure::runQuery(session, query)

query_results$value

## Write the cohort to disk as a PFB

`exportAsPFB()` returns the path invisibly, so you can capture it for downstream steps.

In [ ]:
pfb_path <- picsure::exportAsPFB(session, query, "./temp.avro")

file.info(pfb_path)[, c("size", "mtime")]

## Reading PFB back into R

The picsure package writes the file but doesn't take a position on how you read it back. PFB is plain Avro, so any Avro-capable R library works — `sparklyr::spark_read_avro()`, `arrow::read_arrow()` with the Avro adapter, or shelling out to a Python helper inside the same reticulate env (since `fastavro` rides along with `picsure[pfb]`):

```r
fastavro <- reticulate::import("fastavro")
con      <- reticulate::import_builtins()$open(pfb_path, "rb")
records  <- reticulate::iterate(fastavro$reader(con))
length(records)
```

Pick whichever fits your downstream pipeline.